# Capstone Project — End-to-End Protein ML Pipeline
## Integrating All 17 Modules

## TL;DR — Plain English
**What this notebook does:** Connects everything you learned into one complete pipeline:
sequence → structure prediction → confidence assessment → variant effect → ΔΔG fine-tuning → deployment.

**This is the interview simulation.** A real ML engineer at computational biology ML teams does exactly this:
given a protein of interest, run the full computational pipeline to inform drug design.

**Time:** 4-6 hours | **Prerequisites:** All modules 00-16


## Capstone — Quick Reference for Beginners

| Term | Plain English |
|------|---------------|
| **EGFR** | Epidermal Growth Factor Receptor — a kinase mutated in ~15% of lung cancers; targeted by erlotinib |
| **kinase** | Enzyme that adds phosphate groups to other proteins — switches them on or off |
| **L858R** | Most common EGFR activating mutation; leucine (L) at position 858 changed to arginine (R) |
| **T790M** | EGFR resistance mutation — emerges after first-generation inhibitor treatment |
| **ESM-2** | Meta's protein language model — converts amino acid sequence to rich embedding vectors |
| **delta-delta-G (ddG)** | Free energy change of mutation: negative = stabilising, positive = destabilising |
| **mutation scanning** | Test all 19 substitutions at every position — identifies hot-spot residues |
| **conformal prediction** | Gives guaranteed-coverage prediction intervals (e.g. 95% CI on ddG) |
| **Bayesian optimisation** | Efficient search strategy: model uncertainty -> pick most informative next experiment |
| **pLDDT** | AlphaFold's confidence score per residue (0-100) — low = disordered or unreliable |
| **FastAPI endpoint** | Web URL that accepts a mutation and returns a ddG prediction in real time |
| **executive summary** | 200-word plain-English description of your findings written for a non-technical audience |

## 🟢 How to Use This Capstone Notebook

**This is the FINAL notebook.** If you've done all 17 modules, run each cell top to bottom — you should recognise every function from a previous module.

If you haven't completed all modules yet, that's OK — treat each step as a **preview** and come back after completing the prerequisites listed in the pipeline diagram below.

| Step | Prerequisite modules |
|------|---------------------|
| 1. Sequence validation | 00, 01 |
| 2. Domain analysis | 01 |
| 3. ESM-2 embeddings | 15 |
| 4. Structure confidence | 07/03 |
| 5. ΔΔG prediction | 10/01 |
| 6. Uncertainty | 13/01 |

> **Interview readiness:** After running this notebook end-to-end, time yourself explaining the pipeline in 3 minutes. If you can do that without notes, you are ready.

---
## The Pipeline You Will Build

```
INPUT: Protein sequence (human EGFR kinase domain, a cancer drug target)
           │
           ▼
    Step 1: Parse & validate sequence (Module 00/01)
           │
           ▼
    Step 2: Sequence analysis — find domains, signal peptides (Module 01)
           │
           ▼
    Step 3: Build ML features — ESM-2 embeddings (Module 15)
           │
           ▼
    Step 4: Structure prediction confidence (Module 07/03)
           │
           ▼
    Step 5: ΔΔG prediction for known mutations (Module 10/01)
           │
           ▼
    Step 6: Uncertainty quantification — which predictions to trust? (Module 13)
           │
           ▼
OUTPUT: Ranked list of mutations with predicted binding effect + confidence
```

Each step uses concepts and code from the corresponding module.


## 🟢 Complete Beginner's Guide

**Assumed background:** Has completed all 17 modules. Now ready to integrate everything into a single pipeline.

### What this notebook does

This capstone builds an end-to-end EGFR mutation effect prediction pipeline that uses:
- Module 00: Python/data loading
- Module 01–02: Sequence parsing, variant annotation
- Module 03: Structural analysis, RMSD computation
- Module 05–06: Deep learning and GNN-based features
- Module 07/10: AlphaFold3/OpenFold structure prediction integration
- Module 13: Uncertainty quantification on predictions
- Module 16: MLOps — logging, monitoring, deployment

### How to approach this notebook

Think of it as a **systems integration test**. Each section calls a component you built in a previous module. Your job here is not to learn new concepts — it is to see how they compose into a working system.

### What to do if a step fails

1. **Read the error message fully** — it tells you which module is breaking.
2. **Go back to that module's notebook** — re-run it standalone to verify it works.
3. **Check data shapes** — most failures are shape mismatches between pipeline stages.
4. **Use the fallback data** — each section has a simulated fallback; use it to continue and come back to fix the real data later.
5. **Isolate the failing cell** — run sections before and after the failure to narrow down where the break is.

### Vocabulary you MUST know

| Term | One-line definition |
|------|--------------------|
| `EGFR` | Epidermal Growth Factor Receptor — a kinase mutated in many lung cancers |
| `ΔΔG` | Change in binding free energy caused by a mutation |
| `pipeline` | A sequence of processing steps where output of one feeds next |
| `end-to-end` | From raw input (sequence/mutation) to final prediction (effect score) |

### 3-Step Reading Strategy for Beginners

1. **Read all markdown cells first** — draw the pipeline diagram on paper.
2. **Run code cells one at a time** — print the data shape at each stage transition.
3. **Modify one mutation and re-run** — change the EGFR variant and trace how all downstream predictions change.

### If you're stuck

- **YouTube:** 'How to structure a machine learning project' by Full Stack Deep Learning.
- **Book:** *Designing Machine Learning Systems* by Chip Huyen — Chapter on ML pipelines.
- **Web:** https://clinicalgenome.org/affiliation/50013/ — EGFR variant curation resources.


## 📖 Prerequisites & Cross-References

**Before this notebook, you should be comfortable with:**
- **ESM-2 embeddings** — protein language model representations. *Review: `15_self_supervised_learning/01_contrastive_ssl.ipynb`*
- **ΔΔG prediction** — stability change prediction from structure. *Review: `10_openfold3_finetuning/01_protein_structure_finetuning.ipynb`*
- **Uncertainty quantification** — MC Dropout, calibration. *Review: `13_bayesian_methods/01_bayesian_ml_uncertainty.ipynb`*
- **Bayesian optimization** — acquisition functions, surrogate models. *Review: `13_bayesian_methods/01_bayesian_ml_uncertainty.ipynb`*
- **FastAPI deployment** — serving ML models via REST API. *Review: `16_mlops_deployment/01_mlops_for_protein_ml.ipynb`*
- **AlphaFold 3 structure** — Pairformer, confidence heads. *Review: `07_alphafold3_core/01_af3_architecture.ipynb`*

**Quick recap of terms used here:**
- **embedding** — ESM-2 converts each residue to a 480/1280-dim vector.
- **one-hot** — binary encoding (20-dim for amino acids).
- **batch size** — number of sequences processed together. *See `15/01` for ESM-2, `05/01` for basics.*

In [ ]:
# STEP 1: Sequence Validation and Analysis
# From Module 00/01 (Python Core for Bioinformatics) + Module 01/01 (Alignment)

import numpy as np
import torch

# EGFR kinase domain (real sequence, truncated to 50 aa for demo)
# UniProt P00533, residues 712-761
EGFR_KINASE = "KVLGSGAFGTVYKGLWIPEGEKVKIPVAIKELREATSPKANKEILDEAYVMASVDNPHVCRLLGICLTSTVQLITQLMPFGCLLDYVREHKDNIGSQYLLNWCVQIAKGMNYLEDRRLVHRDLAARNVLVKTPQHVKITDFGLAKLLGAEEKEYHAEGGKVPIKWMALESILHRIYTHQSDVWSYGVTVWELMTFGSKPYDGIPASEISSILEKGERLPQPPICTIDVYMIMVKCWMIDADSRPKFRELIIEFSKMARDPQRYLVIQGDERMHLPSPTDSNFYRALMDEEDMDDVVDADEYLIPQQGFFSSPSTSRTPLLSSLSATSNNSTVACIDRNGLQSCPIKEDSFLQRYSSDPTGALTEDSIDDTFLPVPEYINQSVPKRPAGSVQNPVYHNQPLNPAPSRDPHYQDPHSTAVGNPEYLNTVQPTCVNSTFDSPAHWAQKGSHQISLDNPDYQQDFFPKEAKPNGIFKGSTAENAEYLRVAPQSSEFIGA"
EGFR_SHORT  = EGFR_KINASE[:50]

AMINO_ACIDS = 'ACDEFGHIKLMNPQRSTVWY'
AA_PROPS = {
    'hydrophobic': set('AILMFWVP'),
    'polar':       set('STNQ'),
    'charged_pos': set('KRH'),
    'charged_neg': set('DE'),
    'aromatic':    set('FWY'),
}

def validate_sequence(seq):
    invalid = set(seq) - set(AMINO_ACIDS)
    if invalid:
        return False, f"Invalid amino acids: {invalid}"
    return True, f"Valid: {len(seq)} residues"

def analyze_composition(seq):
    counts = {aa: seq.count(aa)/len(seq) for aa in AMINO_ACIDS}
    # Group properties
    result = {}
    for prop, aas in AA_PROPS.items():
        result[prop] = sum(counts.get(aa, 0) for aa in aas)
    result['mw_estimate_kDa'] = len(seq) * 110 / 1000  # ~110 Da per residue
    return result

valid, msg = validate_sequence(EGFR_SHORT)
comp = analyze_composition(EGFR_SHORT)
print(f"Sequence: {EGFR_SHORT}")
print(f"Validation: {msg}")
print(f"Composition:")
for k, v in comp.items():
    print(f"  {k:<20}: {v:.3f}")

## Self-Assessment Rubric

Use this checklist to evaluate whether your capstone meets interview-ready quality.

### Technical Checklist
- [ ] ESM-2 embeddings computed correctly (shape: L x 480 or L x 1280)
- [ ] ddG predictions produced for all 19 substitutions at >=3 clinically relevant positions
- [ ] Uncertainty bounds computed using bootstrap CI or conformal prediction
- [ ] Bayesian optimisation loop runs >=5 iterations and identifies top mutations
- [ ] Clinical ranking table produced with top 5 mutations annotated with known oncogenic status
- [ ] All code cells run end-to-end without manual intervention

### Communication Checklist
- [ ] Executive summary written in plain English (<200 words, no jargon)
- [ ] Every plot has axis labels, title, and units
- [ ] Methods section cites primary papers with DOIs (AF3: Abramson et al. 2024, ESM-2: Lin et al. 2023)
- [ ] Results section distinguishes between model predictions and experimentally validated mutations
- [ ] Limitations section acknowledges what the model cannot predict (e.g. post-translational modifications)

### Scoring Guide
| Score | Level | What it means |
|-------|-------|---------------|
| 10-11 | Portfolio-ready | Ready to present at an interview or include in a job application |
| 7-9 | Solid | Good understanding; revisit the unchecked items |
| 4-6 | Needs work | Core pipeline works but missing key components |
| 0-3 | Restart | Re-read modules 13, 15, 16 before retrying the capstone |

## 🎤 Interview Questions

**Q1 (Easy):** What is EGFR and why is it a clinically important drug target?
<details><summary>Answer</summary>
EGFR (Epidermal Growth Factor Receptor) is a receptor tyrosine kinase that controls cell growth and division. Activating mutations in EGFR drive approximately 15-20% of non-small cell lung cancers. It is targeted by tyrosine kinase inhibitors (erlotinib, gefitinib, osimertinib), making it one of the most important examples of precision oncology where specific mutations guide treatment selection.
</details>

**Q2 (Easy):** What are L858R and T790M mutations and why do they matter for treatment?
<details><summary>Answer</summary>
L858R is an activating mutation in the EGFR kinase domain that makes tumours sensitive to first-generation TKIs like erlotinib. T790M is a resistance mutation that emerges during treatment, blocking drug binding by introducing a bulkier residue at the gatekeeper position. This drove the development of third-generation inhibitors like osimertinib, which can overcome T790M resistance. Understanding these mutations is central to EGFR treatment sequencing.
</details>

**Q3 (Easy):** What does the end-to-end pipeline produce as its final output?
<details><summary>Answer</summary>
The pipeline takes an EGFR protein sequence as input and produces: predicted DDG values for clinically relevant mutations, uncertainty quantification via conformal prediction intervals, structural impact visualisation, and a ranked list of mutations prioritised by Bayesian optimisation for experimental follow-up. The output integrates sequence analysis, structure prediction, stability prediction, and clinical interpretation.
</details>

**Q4 (Medium):** Why do we use ESM-2 embeddings instead of one-hot encoding in the pipeline?
<details><summary>Answer</summary>
ESM-2 embeddings capture evolutionary conservation, structural context, and long-range dependencies learned from 250 million protein sequences. One-hot encoding treats each amino acid position independently with no contextual information. For DDG prediction, the local structural environment around a mutation matters enormously — ESM-2 embeddings encode this context implicitly, improving prediction accuracy by 10-30% on benchmark datasets compared to one-hot baselines. The embeddings also transfer well to EGFR even if EGFR-specific training data is limited.
</details>

**Q5 (Medium):** How does conformal prediction help a clinician interpret DDG predictions?
<details><summary>Answer</summary>
Conformal prediction provides prediction intervals with guaranteed finite-sample coverage — for example, "the DDG for this mutation is -2.1 kcal/mol with a 90% prediction interval of [-3.0, -1.2]." This is more actionable than a point estimate because clinicians can assess whether the entire interval falls in the destabilising range (indicating confident drug sensitivity) or spans both stabilising and destabilising values (indicating uncertainty that warrants additional testing). The coverage guarantee holds regardless of the underlying model, which is critical for clinical decision-making.
</details>

**Q6 (Medium):** Design the Bayesian optimisation loop for prioritising EGFR mutations to test.
<details><summary>Answer</summary>
Surrogate model: Gaussian process with a Matern kernel over ESM-2 mutation embeddings, trained on the initial set of experimentally measured DDG values. Acquisition function: Expected Improvement over the current best destabilising mutation. Each iteration: evaluate the acquisition function over all candidate mutations (single and double mutants within the kinase domain), select the top-k mutations (batch size matched to experimental throughput), measure DDG experimentally, update the GP, and repeat. Add a diversity penalty to avoid testing clustered mutations. Stop when EI falls below a threshold or the experimental budget is exhausted.
</details>

**Q7 (Medium):** What validation would you perform before deploying this pipeline in a clinical setting?
<details><summary>Answer</summary>
(1) Retrospective validation: test predictions against all known COSMIC EGFR mutations with clinical annotations. (2) Calibration assessment: verify that conformal prediction intervals achieve stated coverage on a held-out set of experimentally measured DDG values. (3) Out-of-distribution detection: ensure the pipeline flags mutations in regions not covered by training data. (4) Comparison with existing tools: benchmark against established predictors (FoldX, Rosetta, EVE). (5) Prospective pilot: partner with a clinical lab to predict outcomes for a blinded set of new patient mutations. (6) Regulatory: document all training data, model versions, and validation results for potential FDA software-as-medical-device review.
</details>

**Q8 (Hard):** How would you extend this pipeline to predict drug resistance mutations prospectively?
<details><summary>Answer</summary>
Systematically enumerate all possible single and double mutations in the drug-binding pocket. For each, predict DDG of drug binding (not just protein stability) using a binding-aware DDG model or molecular docking scores. Flag mutations where the predicted DDG of drug binding is significantly positive (weakened binding) while protein stability DDG is near-neutral (the protein remains functional). Rank candidates by the product of resistance likelihood and evolutionary accessibility (using ESM-2 log-likelihoods as a proxy for mutational probability). Validate the top predictions against known resistance mutations from clinical databases, then publish prospective predictions for mutations not yet observed clinically.
</details>

**Q9 (Hard):** Compare your pipeline's predictions with known COSMIC mutation data — how would you benchmark?
<details><summary>Answer</summary>
Download COSMIC EGFR mutations with clinical significance annotations (pathogenic, benign, drug-resistant). Split into training and held-out test sets stratified by mutation type (missense, in-frame deletion). Metrics: (1) Classification — AUROC and AUPRC for distinguishing pathogenic from benign mutations using predicted DDG as the score. (2) Regression — Spearman correlation between predicted DDG and experimental DDG values where available (from literature or ProTherm/ThermoMutDB). (3) Ranking — Normalized Discounted Cumulative Gain (NDCG) for ranking drug-resistance mutations. (4) Calibration — fraction of COSMIC mutations falling within predicted conformal intervals. Compare against FoldX, Rosetta ddg_monomer, and EVE evolutionary model. Report confidence intervals on all metrics via bootstrapping.
</details>

**Q10 (Hard):** Present this pipeline to a non-technical oncology team in 5 minutes — what do you show?
<details><summary>Answer</summary>
Slide 1 (1 min): The problem — patient has an EGFR mutation, which drug will work? Show current workflow (send to sequencing lab, wait, manual literature lookup). Slide 2 (1 min): The solution — our pipeline takes the mutation and returns a prediction with confidence bounds in seconds. Show the input/output interface. Slide 3 (1.5 min): Validation — "we tested on 200 known EGFR mutations and correctly identified 92% of resistance cases." Show a simple confusion matrix and one case study where the pipeline correctly predicted T790M resistance. Slide 4 (1 min): Uncertainty — "when the model is unsure, it tells you" — show a prediction with a wide confidence interval flagged for additional testing. Slide 5 (0.5 min): Next steps — clinical pilot with your patient cohort, integration with your EMR system, timeline and cost. No equations, no architecture diagrams, no jargon. Focus on patient outcomes and clinical workflow improvement.
</details>

> 📋 **Expected output:** The EGFR protein sequence (~1210 amino acids) starting with `MRPSGTAGAA...` fetched from UniProt accession P00533.
> ✅ If you see the sequence printed, your internet connection and UniProt API are working.
> ❌ If you see a timeout error, check your internet or try again in 30 seconds.